## Splitting and ingesting the content of various URLs (across UK destinations)

In [1]:
# Preparing the Chroma DB collections

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=ollama_embeddings,
)

uk_granular_collection.reset_collection() #A

In [2]:
# Splitting and ingesting HTML content with the HTMLSectionSplitter

from langchain_text_splitters import HTMLSectionSplitter
from langchain_community.document_loaders import AsyncHtmlLoader


uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]
headers_to_split_on = [("h1", "Header 1"),("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(headers_to_split_on=headers_to_split_on)


def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #B
        temp_chunks = html_section_splitter.split_text(
            html_string) #C
        h2_temp_chunks = [chunk for chunk in 
                          temp_chunks if "Header 2" 
                          in chunk.metadata] #D
        all_chunks.extend(h2_temp_chunks) 

    return all_chunks


for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url) #E
    docs =  html_loader.load() #F
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(
            documents=granular_chunks)

#A In case it exists
#B Extract the HTML text from the document
#C Each chunk is a H1 or H2 HTML section
#D Only keep content associated with H2 sections        
#E Loader for one destination
#F Documents of one destination

USER_AGENT environment variable not set, consider setting it to identify your requests.
Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.10it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.26it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.68it/s]


{'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  2.30it/s]


{'source': 'https://en.wikivoyage.org/wiki/West_Cornwall', 'title': 'West Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


## Rewrite-retrieve-read

In [3]:
# Retrieving content with original user question

user_question = "Tell me some fun things I can enjoy in Cornwall"
initial_results = uk_granular_collection.similarity_search(query=user_question,k=4)
for doc in initial_results:
    print(doc)

page_content='Contents 
 
 
 
 
 
 
 
 1   Regions 
 
 
 
 
 
 
 2   Towns and cities 
 
 
 
 
 
 
 3   Other destinations 
 
 
 
 
 
 
 4   Understand 
 
 
 
 
 4.1   Visitor information 
 
 
 
 
 
 
 
 
 5   Talk 
 
 
 
 
 5.1   English 
 
 
 
 
 
 
 5.2   Cornish 
 
 
 
 
 
 
 
 
 6   Get in 
 
 
 
 
 6.1   By plane 
 
 
 
 
 
 
 6.2   By ferry 
 
 
 
 
 
 
 6.3   By train 
 
 
 
 
 
 
 6.4   By car 
 
 
 
 
 
 
 6.5   By coach 
 
 
 
 
 
 
 
 
 7   Get around 
 
 
 
 
 7.1   By bus 
 
 
 
 
 
 
 7.2   By train 
 
 
 
 
 
 
 7.3   By ferry/boat 
 
 
 
 
 
 
 
 
 8   See 
 
 
 
 
 8.1   National Trust properties 
 
 
 
 
 
 
 8.2   National Trust gardens 
 
 
 
 
 
 
 
 
 9   Do 
 
 
 
 
 
 
 10   Eat 
 
 
 
 
 10.1   Savoury 
 
 
 
 
 
 
 10.2   Sweet 
 
 
 
 
 
 
 
 
 11   Drink 
 
 
 
 
 11.1   Ale & beer 
 
 
 
 
 
 
 11.2   Cider 
 
 
 
 
 
 
 11.3   Wine 
 
 
 
 
 
 
 11.4   Mead 
 
 
 
 
 
 
 11.5   Spirits 
 
 
 
 
 
 
 
 
 12   Festivals 
 
 
 
 
 
 
 13   Sleep 
 
 
 
 
 
 

In [4]:
# Question rewrite

# Setting up the query rewriter chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

rewriter_prompt_template = """
Generate search query for the Chroma DB vector store
from a user question, allowing for a more accurate 
response through semantic search.
Just return the revised Chroma DB query, with quotes around it. 

User question: {user_question}
Revised Chroma DB query:
"""

rewriter_prompt = ChatPromptTemplate.from_template(
    rewriter_prompt_template) 
rewriter_chain = rewriter_prompt | llm | StrOutputParser()

# Retrieving content with the rewritten query
user_question ="Tell me some fun things I can do in Cornwall"

search_query = rewriter_chain.invoke({"user_question": user_question})
print(search_query)

improved_results = uk_granular_collection.similarity_search(query=search_query,k=3)
for doc in improved_results:
    print(doc)

"activities and tourist attractions in Cornwall, fun things to do, sightseeing, beaches, landmarks, and local experiences in Cornwall"
page_content='Contents 
 
 
 
 
 
 
 
 1   Towns and villages 
 
 
 
 
 
 
 2   Other destinations 
 
 
 
 
 
 
 3   Understand 
 
 
 
 
 
 
 4   Get in 
 
 
 
 
 4.1   By train 
 
 
 
 
 
 
 4.2   By car 
 
 
 
 
 
 
 4.3   By plane 
 
 
 
 
 
 
 
 
 5   Get around 
 
 
 
 
 5.1   By bus 
 
 
 
 
 
 
 5.2   By train 
 
 
 
 
 
 
 
 
 6   See 
 
 
 
 
 6.1   National Trust properties 
 
 
 
 
 
 
 
 
 7   Do 
 
 
 
 
 7.1   Festivals 
 
 
 
 
 
 
 
 
 8   Drink 
 
 
 
 
 
 
 9   Stay safe 
 
 
 
 
 
 
 10   Go next 
 
 
 
 
 
 
 
 
 
 
 
 
 
 North Cornwall  is in  Cornwall . It includes much of the Cornish coast along the Celtic Sea and some top surfing areas.' metadata={'Header 2': 'Contents'}
page_content='Contents 
 
 
 
 
 
 
 
 1   Towns and villages 
 
 
 
 
 
 
 2   Other destinations 
 
 
 
 
 
 
 3   Understand 
 
 
 
 
 
 
 4   Get in 
 
 
 


In [5]:
# Combining everything in a single RAG chain

from langchain_core.runnables import RunnablePassthrough


retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(
    rag_prompt_template) 

rewrite_retrieve_read_rag_chain = (
    {
        "context": {"user_question": RunnablePassthrough()} 
            | rewriter_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the rewritten query
#B This is the original user question


user_question = "Tell me some fun things I can do in Cornwall"

answer = rewrite_retrieve_read_rag_chain.invoke(user_question)
print(answer)

Based on the documents provided, you can do the following in Cornwall:

*   **Attend Festivals:** There are specific sections dedicated to festivals in various parts of the county.
*   **Visit National Trust Properties:** You can see various National Trust properties and gardens.
*   **Explore Cultural and Historical Sites:** You can explore the county's wealth of archaeology, its mining heritage (recognized by UNESCO), and its diverse Celtic heritage.
*   **Visit the Eden Project:** Located near St. Austell in Mid-Cornwall, you can visit the biomes.
*   **Enjoy the Coastline:** You can visit the Cornish coast along the Celtic Sea and the English Channel, including top surfing areas and the celebrated Land's End.


## Multiple query generation with MultiQueryRetriever

In [6]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate

from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

# Implementing a custom MultiQueryRetriver

# Setting up the prompt
multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task 
is to generate five different versions of the given 
user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user 
question, your goal is to help the user overcome some of 
the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.
Original question: {question}
"""

multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template) 

# Setting up the multi-query parser
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  

questions_parser = LineListOutputParser()

# Setting up the chain to generate multiple queries
llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)
multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

# Testing the Multi query gen chain
user_question = "Tell me some fun things I can do in Cornwall"

multiple_queries = multi_query_gen_chain.invoke(user_question)
multiple_queries

['What are the best tourist attractions and activities in Cornwall?',
 'List of outdoor adventures and hidden gems to visit in Cornwall.',
 'What are some family-friendly things to do in Cornwall?',
 'Top-rated experiences and sightseeing spots in Cornwall, UK.',
 'What are the most popular and unique things to see and do in Cornwall?']

In [7]:
# Setting up the MultiQueryRetriever
basic_retriever = uk_granular_collection.as_retriever()

multi_query_retriever = MultiQueryRetriever(
    retriever=basic_retriever, llm_chain=multi_query_gen_chain, 
    parser_key="lines" #A
)  
#A this is the key for the parsed output

# Using the multi_query retriever
user_question = "Tell me some fun things I can do in Cornwall"

retrieved_docs = multi_query_retriever.invoke(user_question)
retrieved_docs

[Document(id='273048c2-482c-401c-9b2b-5b532ffdebee', metadata={'Header 2': 'Contents'}, page_content='Contents \n \n \n \n \n \n \n \n 1   Regions \n \n \n \n \n \n \n 2   Towns and cities \n \n \n \n \n \n \n 3   Other destinations \n \n \n \n \n \n \n 4   Understand \n \n \n \n \n 4.1   Visitor information \n \n \n \n \n \n \n \n \n 5   Talk \n \n \n \n \n 5.1   English \n \n \n \n \n \n \n 5.2   Cornish \n \n \n \n \n \n \n \n \n 6   Get in \n \n \n \n \n 6.1   By plane \n \n \n \n \n \n \n 6.2   By ferry \n \n \n \n \n \n \n 6.3   By train \n \n \n \n \n \n \n 6.4   By car \n \n \n \n \n \n \n 6.5   By coach \n \n \n \n \n \n \n \n \n 7   Get around \n \n \n \n \n 7.1   By bus \n \n \n \n \n \n \n 7.2   By train \n \n \n \n \n \n \n 7.3   By ferry/boat \n \n \n \n \n \n \n \n \n 8   See \n \n \n \n \n 8.1   National Trust properties \n \n \n \n \n \n \n 8.2   National Trust gardens \n \n \n \n \n \n \n \n \n 9   Do \n \n \n \n \n \n \n 10   Eat \n \n \n \n \n 10.1   Savoury \n \n \

In [8]:
# Using directly a standard MultiQueryRetriever
std_multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=basic_retriever, llm=llm
)
user_question = "Tell me some fun things I can do in Cornwall"

retrieved_docs = std_multi_query_retriever.invoke(user_question)
retrieved_docs

[Document(id='273048c2-482c-401c-9b2b-5b532ffdebee', metadata={'Header 2': 'Contents'}, page_content='Contents \n \n \n \n \n \n \n \n 1   Regions \n \n \n \n \n \n \n 2   Towns and cities \n \n \n \n \n \n \n 3   Other destinations \n \n \n \n \n \n \n 4   Understand \n \n \n \n \n 4.1   Visitor information \n \n \n \n \n \n \n \n \n 5   Talk \n \n \n \n \n 5.1   English \n \n \n \n \n \n \n 5.2   Cornish \n \n \n \n \n \n \n \n \n 6   Get in \n \n \n \n \n 6.1   By plane \n \n \n \n \n \n \n 6.2   By ferry \n \n \n \n \n \n \n 6.3   By train \n \n \n \n \n \n \n 6.4   By car \n \n \n \n \n \n \n 6.5   By coach \n \n \n \n \n \n \n \n \n 7   Get around \n \n \n \n \n 7.1   By bus \n \n \n \n \n \n \n 7.2   By train \n \n \n \n \n \n \n 7.3   By ferry/boat \n \n \n \n \n \n \n \n \n 8   See \n \n \n \n \n 8.1   National Trust properties \n \n \n \n \n \n \n 8.2   National Trust gardens \n \n \n \n \n \n \n \n \n 9   Do \n \n \n \n \n \n \n 10   Eat \n \n \n \n \n 10.1   Savoury \n \n \

## Step-back question

In [9]:
# Setting up the chain to generate the step-back question
llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

step_back_prompt_template = """
    Generate a less specific question (aka Step-back question) 
    for the following detailed question, so that a wider context 
    can be retrieved.
    Detailed question: {detailed_question}
    Step-back question:
"""

step_back_prompt = ChatPromptTemplate.from_template(step_back_prompt_template) 
step_back_question_gen_chain = step_back_prompt | llm | StrOutputParser()

# Testing the step-back-question generation chain
user_question = "Can you give me some tips for a trip to Brighton?"

step_back_question = step_back_question_gen_chain.invoke(user_question)
step_back_question

'Step-back question: **What are the best things to know when planning a trip to a coastal city in the UK?**'

In [ ]:
# Incorporating step-back question generation chain into the RAG chain
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

step_back_question_rag_chain = (
    {
        "context": {"detailed_question": RunnablePassthrough()} 
           | step_back_question_gen_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the step-back question
#B This is the original user question

user_question = "Can you give me some tips for a trip to Brighton?"

answer = step_back_question_rag_chain.invoke(user_question)
print(answer)

I do not know.


In [11]:
user_question = "Can you give me some tips for a trip to West_Cornwall?"

answer = step_back_question_rag_chain.invoke(user_question)
print(answer)

Based on the provided documents, here are some tips and information for a trip to West Cornwall:

**Getting There and Around:**
*   **Getting In:** You can reach the area by train, car, bus, or plane.
*   **Getting Around:** Local transport options include bus and train.

**Things to See and Do:**
*   **National Trust:** You can visit various National Trust properties.
*   **Festivals:** There are festivals to attend in the region.
*   **Geography:** West Cornwall covers the tip of the South West Peninsula, situated between the Celtic Sea to the north and the English Channel to the south, leading out to Land's End.

**General Cornwall Context:**
*   **Heritage:** The area is known for its Celtic heritage, mining heritage (recognized by UNESCO), and archaeology.
*   **Nature:** Over 30% of the county is designated as an


## Hypotetical DocumentEmbeddings (HyDE)

In [12]:
# Setting up the chain to generate the hypotetical document associated to the user question
llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

hyde_prompt_template = """
Write one sentence that could answer the provided question. 
Do not add anything else.
Question: {question}
Sentence:
"""

hyde_prompt = ChatPromptTemplate.from_template(hyde_prompt_template)
hyde_chain = hyde_prompt | llm | StrOutputParser()

# Testing the hyde generation chain
user_question = "What are the best beaches in Cornwall?"

hypotetical_document = hyde_chain.invoke(user_question)
hypotetical_document

'Some of the best beaches in Cornwall include Porthcurnick Beach, Kynance Cove, and Fistral Beach.'

In [13]:
# Incorporating hyde chain into the RAG chain
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
Only use the provided context to answer the question.
If you do not know the answer, just say I do not know. 

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

hyde_rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} 
           | hyde_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the hypotetical document
#B This is the original user question

user_question = "What are the best beaches in Cornwall?"

answer = hyde_rag_chain.invoke(user_question)
print(answer)

The provided context lists several beaches that form an important part of the tourist industry, including Bude, Polzeath, Watergate Bay, Perranporth, Porthtowan, Fistral Beach, Newquay, St Agnes, St Ives, Gyllyngvase beach in Falmouth, and the large beach at Praa Sands.
